# 04 — LLM Response Layer

This notebook validates the first response-generation layer for the Medical Insight Explorer Agent.

The goal is to route natural-language questions to controlled analytics functions, compute real results with pandas, and generate grounded explanations.

This notebook works without an API key by using deterministic fallback explanations.

In [1]:
#Importing
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from agent.data_loader import HealthcareDataLoader
from agent.analytics_engine import HealthcareAnalyticsEngine
from agent.response_generator import ResponseGenerator, QueryRouter

In [2]:
#loading data and initializing components
loader = HealthcareDataLoader(data_dir=PROJECT_ROOT / "data" / "processed")
tables = loader.load_tables()

engine = HealthcareAnalyticsEngine(tables)
response_generator = ResponseGenerator(
    analytics_engine=engine,
    use_llm=False,
)

In [3]:
#Testing router
router = QueryRouter()

test_questions = [
    "Show me the shape of all tables",
    "Give me an inpatient summary",
    "Give me an outpatient summary",
    "What is the average beneficiary age?",
    "Show top inpatient providers",
    "Show top outpatient providers",
    "Show inpatient claims by state",
    "What is the diabetes cost summary?",
    "Can you diagnose these patients?",
]

for question in test_questions:
    print(question, "→", router.route(question))

Show me the shape of all tables → table_shapes
Give me an inpatient summary → inpatient_summary
Give me an outpatient summary → outpatient_summary
What is the average beneficiary age? → age_summary
Show top inpatient providers → top_inpatient_providers
Show top outpatient providers → top_outpatient_providers
Show inpatient claims by state → inpatient_claims_by_state
What is the diabetes cost summary? → diabetes_cost_summary
Can you diagnose these patients? → fallback


In [4]:
#Answering table shape question
response = response_generator.answer_question("Show me the shape of all tables")
response

{'question': 'Show me the shape of all tables',
 'route': 'table_shapes',
 'computed_result': [{'table_name': 'test_beneficiary',
   'rows': 63968,
   'columns': 26},
  {'table_name': 'test_inpatient', 'rows': 9551, 'columns': 30},
  {'table_name': 'test_labels', 'rows': 1353, 'columns': 1},
  {'table_name': 'test_outpatient', 'rows': 125841, 'columns': 27},
  {'table_name': 'train_beneficiary', 'rows': 138556, 'columns': 26},
  {'table_name': 'train_inpatient', 'rows': 40474, 'columns': 30},
  {'table_name': 'train_labels', 'rows': 5410, 'columns': 2},
  {'table_name': 'train_outpatient', 'rows': 517737, 'columns': 27}],
 'explanation': "Route selected: table_shapes. The result was computed using deterministic pandas-based analytics. Computed output: [{'table_name': 'test_beneficiary', 'rows': 63968, 'columns': 26}, {'table_name': 'test_inpatient', 'rows': 9551, 'columns': 30}, {'table_name': 'test_labels', 'rows': 1353, 'columns': 1}, {'table_name': 'test_outpatient', 'rows': 125841,

In [5]:
#Answering inpatient summary
response = response_generator.answer_question("Give me an inpatient summary")
response

{'question': 'Give me an inpatient summary',
 'route': 'inpatient_summary',
 'computed_result': {'total_claims': 40474,
  'unique_beneficiaries': 31289,
  'unique_claims': 40474,
  'unique_providers': 2092,
  'mean_reimbursement': 8960.641893561298,
  'median_reimbursement': 7000.0,
  'total_reimbursement': 362673020.0},
 'explanation': "Route selected: inpatient_summary. The result was computed using deterministic pandas-based analytics. Computed output: {'total_claims': 40474, 'unique_beneficiaries': 31289, 'unique_claims': 40474, 'unique_providers': 2092, 'mean_reimbursement': 8960.641893561298, 'median_reimbursement': 7000.0, 'total_reimbursement': 362673020.0}"}

In [6]:
#Answring age question
response = response_generator.answer_question("What is the average beneficiary age?")
response

{'question': 'What is the average beneficiary age?',
 'route': 'age_summary',
 'computed_result': {'count': 138556,
  'mean_age': 83.10368370911401,
  'median_age': 84.0,
  'min_age': 36.0,
  'max_age': 111.0},
 'explanation': "Route selected: age_summary. The result was computed using deterministic pandas-based analytics. Computed output: {'count': 138556, 'mean_age': 83.10368370911401, 'median_age': 84.0, 'min_age': 36.0, 'max_age': 111.0}"}

In [7]:
#Answering provider question
response = response_generator.answer_question("Show top inpatient providers")
response

{'question': 'Show top inpatient providers',
 'route': 'top_inpatient_providers',
 'computed_result': [{'Provider': 'PRV52019', 'claim_count': 516},
  {'Provider': 'PRV55462', 'claim_count': 386},
  {'Provider': 'PRV54367', 'claim_count': 322},
  {'Provider': 'PRV53706', 'claim_count': 282},
  {'Provider': 'PRV55209', 'claim_count': 275},
  {'Provider': 'PRV56560', 'claim_count': 248},
  {'Provider': 'PRV54742', 'claim_count': 231},
  {'Provider': 'PRV55230', 'claim_count': 225},
  {'Provider': 'PRV52340', 'claim_count': 224},
  {'Provider': 'PRV51501', 'claim_count': 223}],
 'explanation': "Route selected: top_inpatient_providers. The result was computed using deterministic pandas-based analytics. Computed output: [{'Provider': 'PRV52019', 'claim_count': 516}, {'Provider': 'PRV55462', 'claim_count': 386}, {'Provider': 'PRV54367', 'claim_count': 322}, {'Provider': 'PRV53706', 'claim_count': 282}, {'Provider': 'PRV55209', 'claim_count': 275}, {'Provider': 'PRV56560', 'claim_count': 248}

In [8]:
#Answering diabets question
response = response_generator.answer_question("What is the diabetes cost summary?")
response

{'question': 'What is the diabetes cost summary?',
 'route': 'diabetes_cost_summary',
 'computed_result': [{'ChronicCond_Diabetes': 2,
   'count': 8012,
   'mean': 9002.387668497255,
   'median': 7000.0,
   'sum': 72127130},
  {'ChronicCond_Diabetes': 1,
   'count': 32462,
   'mean': 8950.338549688868,
   'median': 7000.0,
   'sum': 290545890}],
 'explanation': "Route selected: diabetes_cost_summary. The result was computed using deterministic pandas-based analytics. Computed output: [{'ChronicCond_Diabetes': 2, 'count': 8012, 'mean': 9002.387668497255, 'median': 7000.0, 'sum': 72127130}, {'ChronicCond_Diabetes': 1, 'count': 32462, 'mean': 8950.338549688868, 'median': 7000.0, 'sum': 290545890}]"}

In [9]:
#Fallback safety test
response = response_generator.answer_question("Can you diagnose these patients?")
response

{'question': 'Can you diagnose these patients?',
 'route': 'fallback',
 'computed_result': {'message': 'This question is not supported by the current analytics routes. Try asking about table shapes, inpatient summary, outpatient summary, age statistics, top providers, state claim counts, or diabetes cost summary.'},
 'explanation': 'This question is not supported by the current analytics routes. Try asking about table shapes, inpatient summary, outpatient summary, age statistics, top providers, state claim counts, or diabetes cost summary.'}

## Optional LLM Mode

If an OpenAI API key is available in `.env`, the response generator can be initialized with:

```python
response_generator = ResponseGenerator(
    analytics_engine=engine,
    use_llm=True,
)


## Cell 12 — Markdown conclusion

```markdown
## Result

The response layer successfully routes natural-language questions to controlled analytics functions.

All numerical results are computed by the deterministic analytics engine before explanation. This creates a grounded response layer for the future Gradio interface.